# Larger-model and document-format sweep
Select an A100 in Runtime > Change runtime type. Run setup and launch once, then monitor. This uses the existing 575 documents; no new data or API keys.
The manifest is `experiments/exploration_20261002.json`: 9B Base, 4B instruct on plain documents, and 4B instruct on documents framed as assistant responses. Each gets rank-32 LoRA, 180 steps, and independent base/final evaluation. Explicit glyph-copy controls are separate from spontaneous transfer.
Do not use Run All: the download cell requires a completed sweep.


In [ ]:
import pathlib, subprocess, sys
repo = pathlib.Path('/content/sdfqwen')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'codex/colab-experiments', '--single-branch',
                    'https://github.com/Beasleydog/sdfqwen.git', str(repo)], check=True)
print(subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True))
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
assert not (repo / '.env').exists(), 'Unexpected .env in cloned repository'
print('Existing documents:', len(list((repo / 'datagen/outputs').glob('*.txt'))))


In [ ]:
from pathlib import Path
import subprocess, sys, datetime, shutil
repo=Path('/content/sdfqwen')
active=subprocess.run(['pgrep','-af','[e]xperiments/(explore_colab|colab_job|train_experiment).py'],capture_output=True,text=True)
if active.stdout.strip():
    print('Already running:',active.stdout)
else:
    subprocess.run(['git','pull','--ff-only'],cwd=repo,check=True)
    old=[repo/n for n in ('colab_status.json','colab_job.log','colab_results.zip') if (repo/n).exists()]
    if old:
        dest=repo/'experiment_results'/('previous_sweep_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
        dest.mkdir(parents=True)
        for p in old: shutil.move(str(p),str(dest/p.name))
    with (repo/'colab_job.log').open('w') as log:
        proc=subprocess.Popen([sys.executable,str(repo/'experiments/explore_colab.py')],cwd=repo,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
    print('Sweep started, PID',proc.pid)


In [ ]:
from pathlib import Path
import json
repo = Path('/content/sdfqwen')
status = repo / 'colab_status.json'
print(status.read_text() if status.exists() else 'Starting')
log = repo / 'colab_job.log'
if log.exists():
    lines = log.read_text(errors='replace').replace('\r','\n').splitlines()
    progress = [line for line in lines if '/120' in line or '/360' in line or '/180' in line]
    events = [line for line in lines if line.startswith(('EVALUATED','COMPLETE',"{'",'Traceback','TypeError','RuntimeError','ImportError'))]
    print('\n'.join(events[-10:] + progress[-1:]))
    if '"state": "failed"' in status.read_text():
        print('\n'.join(lines[-40:]))
print('Evaluation files:', [str(p.relative_to(repo)) for p in (repo/'experiment_results').glob('*/*.json') if p.stem == 'base' or p.stem.startswith('step_')])



In [ ]:
from pathlib import Path
import subprocess
repo=Path('/content/sdfqwen')
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.used,utilization.gpu','--format=csv,noheader'],capture_output=True,text=True).stdout)
print(subprocess.run(['pgrep','-af','[e]xperiments/(explore_colab|colab_job|train_experiment).py'],capture_output=True,text=True).stdout)
lines=(repo/'colab_job.log').read_text(errors='replace').replace('\r','\n').splitlines()
print('\n'.join(lines[-12:]))


In [ ]:
import json
from google.colab import files
status=json.loads((repo/'colab_status.json').read_text())
assert status['state'] in ('complete','complete_with_failures','failed'), 'Training is still active'
files.download(str(repo/'colab_results.zip'))
